# 09 — 2단계 고정 창을 **전체 데이터·확정 백본**으로 확인 (STEP 23)

## 묻는 것 하나

STEP 22 에서 2단계 크롭을 비례 창(`m2.5`) → 고정 창(`f320`)으로 바꾸니
**A4 recall 0.393 → 0.525**, A4→A1 혼동 23.6% → 14.7%, macro-F1 +0.035 였습니다.
사전등록 4기준을 전부 통과했습니다.

**그런데 두 가지가 남았습니다:**

| 한계 | 왜 문제인가 |
|---|---|
| **VL01 만** (전체의 10.8%) | ⚠️ **A4 의 88%는 TL02** 에 있고 거기 A4 는 **절반 크기**입니다 (113px vs 196px). 같은 이득이 날지 모릅니다 |
| **백본이 `effnetv2_s`** | 확정본은 `convnextv2_base`. STEP 22 는 빠른 백본으로 방향만 봤습니다 |

이 노트북이 **둘 다** 없앱니다.

⚠️ **STEP 22 와 절대값을 비교하지 마세요.** 데이터도 백본도 다릅니다.
   판정은 **이 실행 안의 `m2.5` 기준선 대비**로만 합니다.

## 붙일 것 (Add Input)

| 입력 | 필수 |
|---|---|
| `m2.5` 크롭 | ✅ |
| **`f320` 크롭** | ✅ ← **이게 있어야 비교가 됩니다** |
| STEP 16 release | ✅ (매니페스트·설정을 여기서 읽습니다) |

⚠️ 두 태그가 **같은 청크**에 다 있어야 합니다. `--chunk auto` 가 그걸 확인하고
   한쪽만 있는 청크는 빼줍니다 (2026-09-05 에 VL01 이 통째로 빠져 있었습니다).

## 돌리는 법

우측 상단 **[Save Version] → Save & Run All (Commit)**.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. 시간 먼저 — 캐글 세션 안에 들어가나

⚠️ `convnextv2_base` 는 `effnetv2_s` 보다 **4.6배** 느립니다 (STEP 12 실측:
에폭당 10.6분 vs 2.3분). 2판이 세션(9~12시간)을 넘기면 `SUBSET` 을 줄이세요.

**추정치는 믿지 말고** 아래 셀이 실제 GPU 로 재는 값을 보세요 (작업 규칙 1).

In [ ]:
import sys
sys.path.insert(0, DIR)
import torch
from src import crop, env, labels, split, stages, experiments
from src.config import CFG

env.load_prepared()
env.require_gpu()
DEV = "cuda"

# ── 여기만 바꾸면 됩니다 ───────────────────────────────────
TAGS   = ["m2.5", "f320"]      # 기준선이 **먼저**
MODEL  = "convnextv2_base"     # 확정 백본
EPOCHS = 10
SUBSET = 1.0                   # 세션을 넘기면 0.5 → 0.3 으로
# ──────────────────────────────────────────────────────────

mpath = env.work_root() / "manifests" / "manifest_final.parquet"
df = labels.load(mpath)
print(f"{len(df):,}행")
if len(df) < 300_000:
    print("[!] 365,428 보다 훨씬 적습니다 — 옛 데이터일 수 있습니다.")

have = crop.available_tags()
print("붙어 있는 태그:", have)
missing = [t for t in TAGS if t not in have]
if missing:
    raise SystemExit(f"[X] 태그가 없습니다: {missing}. f320 크롭을 Add Input 하세요.")

# 두 태그가 **다 있는** 청크만 씁니다
keep = crop.chunks_with_crops(df, TAGS)
if not keep:
    raise SystemExit("[X] 두 태그가 다 있는 청크가 없습니다.")
df = df[df["chunk"].isin(keep)].reset_index(drop=True)
print(f"\n쓸 청크 {keep} — {len(df):,}행")

s2 = stages.to_stage2(crop.switch_tag(df, TAGS[0], verbose=False))
split.verify(s2, fold=0, strict=True)
tr, va = split.get_fold(s2, CFG().use_fold)
print(f"2단계 train {len(tr):,} → 서브셋 {int(len(tr)*SUBSET):,} / val {len(va):,}")
print("클래스별 val:", va["label"].value_counts().sort_index().to_dict())

# ⚠️ 목록의 튜플은 (모델, **해상도**) 입니다 — 에폭이 아닙니다. `epochs` 는
#    별도 인자이고, (model, EPOCHS) 로 넘기면 해상도가 10 으로 잡히고
#    epochs 누락으로 죽습니다. 03h 가 이미 겪은 것을 제가 또 밟았습니다.
est = experiments.estimate_runtime(
    [MODEL] * len(TAGS), img_size=384,
    n_train=int(len(tr) * SUBSET), epochs=EPOCHS, device=DEV)
print("\n[!] 추정치입니다 — 실측이 아닙니다. 9시간을 넘길 것 같으면 SUBSET 을 줄이세요.")


## 2. 두 판

`m2.5` 가 기준선입니다. **같은 실행 안**에 있어야 판정이 유효합니다 —
다른 세션과 비교하면 실행 간 잡음(±0.02)에 묻힙니다.

In [ ]:
import json
from pathlib import Path
import numpy as np
from src import evaluate, train
from src.config import CLASSES

FOCUS, OTHER = "A4", "A1"
runs = []
for tag in TAGS:
    print(f"\n{'#'*70}\n 판 {tag}\n{'#'*70}")
    view = stages.to_stage2(crop.switch_tag(df, tag, verbose=False))
    split.verify(view, fold=0, strict=True)
    r = experiments.train_and_measure(
        view, stage=2, img_size=384, crop_tag=tag, device=DEV,
        model_name=MODEL, finetune="moderate", aug="default",
        epochs=EPOCHS, subset_frac=SUBSET,
        measure_robust=True, measure_blur=False, n_robust=2000)

    z = np.load(train.ckpt_dir(r["exp_name"]) / "logits_val.npz", allow_pickle=False)
    y, pred = z["y"], z["logits"].argmax(1)
    fi, oi = CLASSES.index(FOCUS), CLASSES.index(OTHER)
    m = y == fi
    _, lo, hi = evaluate.bootstrap_ci(y, pred, metric="recall", cls=fi, n=1000)
    r.update(focus_recall=float((pred[m] == fi).mean()),
             focus_recall_ci=float((hi - lo) / 2),
             f2o_rate=float((pred[m] == oi).mean()),
             n_focus=int(m.sum()))
    print(f"  macro-F1 {r['macro_f1']:.4f} · {FOCUS} recall {r['focus_recall']:.3f}"
          f" (±{r['focus_recall_ci']:.3f}, n={r['n_focus']:,})"
          f" · {FOCUS}->{OTHER} {r['f2o_rate']:.1%}"
          f" · 배율 하락 {r.get('scale_drop', float('nan')):.1%}")
    runs.append(r)


## 3. 판정

`stage2_size_report()` 가 **STEP 22 와 똑같은 기준**으로 판정합니다
(`src/experiments.py` — 노트북 셀이 아니라서 결과를 보고 못 바꿉니다).

In [ ]:
verdict = experiments.stage2_size_report(runs, base_crop="m2.5",
                                        focus=FOCUS, other=OTHER)

out = {"step": "STEP 23 — 2단계 고정 창, 전체 데이터·확정 백본",
       "chunks": keep, "model": MODEL, "epochs": EPOCHS, "subset_frac": SUBSET,
       "n_train": runs[0]["n_train"] if runs else None,
       "step22_vl01_effnetv2s": {"m2.5": {"macro_f1": 0.5520, "A4_recall": 0.393,
                                          "A4_to_A1": 0.236, "scale_drop": 0.279},
                                 "f320": {"macro_f1": 0.5873, "A4_recall": 0.525,
                                          "A4_to_A1": 0.147, "scale_drop": 0.334}},
       "verdict": verdict,
       "runs": [{k: v for k, v in r.items() if k != "report"} for r in runs]}
p = Path("/kaggle/working/step23_stage2_size_full.json")
p.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=float),
             encoding="utf-8")
print(f"\n저장: {p}  ← 이 파일을 공유하세요 (수십 KB)")
print("\n[!] STEP 22 값(위 JSON 의 step22_...)은 **참고용**입니다 —"
      " 데이터도 백본도 달라 절대값 비교 금지.")


## 다음

| 판정 | 다음에 할 것 |
|---|---|
| **채택 후보** | 2단계 크롭을 `f320` 으로 바꾸고 **촬영 가이드 밴드를 다시 뽑습니다** (`robust.usable_range`) — 고정 창은 촬영 거리에 의존하므로 |
| **배율 의존** | 정보는 확인됐지만 배포엔 못 씁니다. 가이드 프레임으로 배율을 강제하는 쪽을 검토 |
| **기각** | STEP 22 는 VL01 특유였던 것. 기록하고 A4 는 다른 축으로 |

어느 쪽이든 `docs/results/STEP23_*.md` 에 남깁니다 — **"VL01 에서만 되더라" 도
결론**이고, 안 적으면 몇 주 뒤에 같은 걸 또 돌립니다.